In [53]:
import geopandas as gpd
import geopandas as gpd
import os
import boto3
from dotenv import load_dotenv
import xml.etree.ElementTree as ET
from pystac_client import Client
import time
import odc.stac
import numpy as np
import matplotlib.pyplot as plt
from pyproj import Transformer
from tqdm import tqdm
from pystac import ItemCollection


import pandas as pd
load_dotenv()



True

In [2]:
FILE = "../data/registros_limpios.geojson"
COPERNICUS_S3_ENDPOINT_URL = "https://eodata.dataspace.copernicus.eu"
CATALOG_URL = "https://stac.dataspace.copernicus.eu/v1"
COLLECTION = "sentinel-2-l1c" # top of atmosphere, vamos a aplicar nuestras propias correciones (ACOLITE)

def copernicus_s3_client():
    s3 = boto3.client(
        "s3",
        endpoint_url=COPERNICUS_S3_ENDPOINT_URL,
        aws_access_key_id=os.environ["CDSE_S3_ACCESS_KEY"],
        aws_secret_access_key=os.environ["CDSE_S3_SECRET_KEY"],
    )
    return s3

BAND_RESOLUTION = { "B01": 60, "B02": 10, "B03": 10, "B04": 10, "B05": 20, "B06": 20, "B07": 20, "B08": 10, "B8A": 20, "B09": 60, "B10": 60, "B11": 20, "B12": 20}

def bands_downsample(bands: list[str]):
    res = -float("inf")
    for b in bands:
        res = max(res, BAND_RESOLUTION[b])
    return res
    
def bands_upsample(bands: list[str]):
    res = -float("inf")
    for b in bands:
        res = min(res, BAND_RESOLUTION[b])
    return res

CLUSTERS = [
    "RN-UPM1",
    "RN-UPM2",
    "RDP-MONTES",
    "LDS"
]

In [3]:
to_wgs84 = Transformer.from_crs("EPSG:32721", "EPSG:4326", always_xy=True)
gdf = gpd.read_file(FILE)
s3 = copernicus_s3_client()
catalogo = Client.open(CATALOG_URL)

In [4]:
gdf.shape

(3089, 6)

In [5]:
gdf[gdf["grupo_nombre"] == "LDS"].total_bounds

array([ 668680.46418811, 6142737.1530541 ,  679296.97302174,
       6153076.9715142 ])

#### Extremadamento lento

In [6]:
#for cluster in CLUSTERS:

#    cluster_points = gdf[gdf["grupo_nombre"] == cluster]
#    cluster_bounds = cluster_points.total_bounds
#    xmin, ymin, xmax, ymax = cluster_bounds
#
#    xmin, ymin = to_wgs84.transform(xmin, ymin)
#    xmax, ymax = to_wgs84.transform(xmax, ymax)
#
#    bbox = [xmin, ymin, xmax, ymax]
#
#    #print(gdf["fecha"].min())
#
#    min_fecha = cluster_points["fecha"].min().strftime("%Y-%m-%d")
#    max_fecha = cluster_points["fecha"].max().strftime("%Y-%m-%d")
#
#    search = catalogo.search(
#        collections=[COLLECTION],
#        datetime=f"{min_fecha}/{max_fecha}",
#        bbox=bbox,
#    )
#    #print(items)
#    break
#search.matched()

In [7]:
months_with_samples = gdf[gdf["grupo_nombre"] == "RN-UPM1"]["fecha"].dt.to_period("M").unique()
months_with_samples

<PeriodArray>
['2017-01', '2017-03', '2017-11', '2018-01', '2018-03', '2018-05', '2018-09',
 '2019-01', '2019-03', '2019-05', '2019-07', '2019-09', '2019-11', '2020-01',
 '2020-03', '2020-05', '2020-09', '2020-11', '2021-03', '2021-09', '2021-11',
 '2022-02', '2022-09', '2022-12', '2023-12', '2024-03', '2024-12', '2025-03']
Length: 28, dtype: period[M]

In [8]:
months_with_samples[0].start_time

Timestamp('2017-01-01 00:00:00')

In [9]:
months_with_samples[0].end_time

Timestamp('2017-01-31 23:59:59.999999')

In [10]:
# no es muy importante esto
# como la API tira Rate Limit a veces (cuando hacemo demasiados requests, es una medidad de proteccion que tiene)
# esto es una configuracion para que nuestro codigo sea resiliente contra eso, y vuelva a intentar y no se crashee
from urllib3.util.retry import Retry
from pystac_client.stac_api_io import StacApiIO
retry = Retry(
    total=5,
    backoff_factor=2,          # waits 0, 2, 4, 8, 16s between retries
    status_forcelist=[429, 502, 503, 504],
    respect_retry_after_header=True,
)
stac_io = StacApiIO(max_retries=retry)
catalogo = Client.open(CATALOG_URL, stac_io=stac_io)

### Mas rapido?

In [11]:
#for cluster in CLUSTERS:
#    cluster_points = gdf[gdf["grupo_nombre"] == cluster]
#    cluster_bounds = cluster_points.total_bounds
#    xmin, ymin, xmax, ymax = cluster_bounds
#
#    xmin, ymin = to_wgs84.transform(xmin, ymin)
#    xmax, ymax = to_wgs84.transform(xmax, ymax)
#
#    tol = 0.001 # tolerancia para puntos que pueden llegar a estar al bordex de cluster_bounds, 100m mas o menos en grados
#    bbox = [xmin - tol, ymin - tol, xmax + tol, ymax + tol]
#
#    #print(gdf["fecha"].min())
#
#    months_with_samples = cluster_points["fecha"].dt.to_period("M").unique()
#    items_all = []
#
#    for month in tqdm(months_with_samples, desc=f"Searching months of {cluster}"):
#        start = month.start_time.strftime("%Y-%m-%d")
#        end = month.end_time.strftime("%Y-%m-%d")
#        #print(start, end)
#        search = catalogo.search(
#            collections=[COLLECTION],
#            datetime=f"{start}/{end}",
#            bbox=bbox,
#        )
#        time.sleep(1)
#        items_all.extend(search.items())
#    break

In [12]:
#items_all[0].datetime.strftime("%Y-%m-%d %H:%M:%S")

In [13]:
#months_with_samples[:10]

#### Algoritmo de matchups
1. Recorrer todos los meses que sabemos que hay muestras
2. para cada mes_i:
3. agarrar las fechas de las muestras dentro de ese mes i
4. hacer un for de 0 a 6 con la fecha en comparación con la fechas que se obtuvieron del satelite para +- 5días.
5. hacer otro data_geo y si hay un match guardar el id del item en una nueva columna.
6. si no hay match no guardar anda porque sabemos que se pasaron de los +- 5 días.

#### 

In [14]:
#cluster_points
def compute_matchups(cluster_points, items):
    matchups = []
    for index, fila in cluster_points.iterrows():
        fecha = fila["fecha"]
        best_diff = pd.Timedelta(days=5)   # threshold AND reset, per sample
        best_item = None

        for item in items:
            fecha_satelite = pd.Timestamp(item.datetime).tz_localize(None)
            diff = abs(fecha_satelite - fecha)     # abs on the Timedelta
            if diff <= best_diff:
                best_diff = diff
                best_item = item

        if best_item is not None:
            matchups.append((best_item, index, best_diff))   # best_item, not item
    return matchups

In [19]:
all_matchups = []
for cluster in CLUSTERS:
    cluster_points = gdf[gdf["grupo_nombre"] == cluster]
    cluster_bounds = cluster_points.total_bounds
    xmin, ymin, xmax, ymax = cluster_bounds

    xmin, ymin = to_wgs84.transform(xmin, ymin)
    xmax, ymax = to_wgs84.transform(xmax, ymax)

    tol = 0.001 # tolerancia para puntos que pueden llegar a estar al bordex de cluster_bounds, 100m mas o menos en grados
    bbox = [xmin - tol, ymin - tol, xmax + tol, ymax + tol]

    #print(gdf["fecha"].min())

    months_with_samples = cluster_points["fecha"].dt.to_period("M").unique()
    items_by_id = {}

    for month in tqdm(months_with_samples, desc=f"Searching months of {cluster}"):
        # padding +-5 dias para no perder matches de muestras cerca del limite del mes
        start = (month.start_time - pd.Timedelta(days=5)).strftime("%Y-%m-%d")
        end = (month.end_time + pd.Timedelta(days=5)).strftime("%Y-%m-%d")
        #print(start, end)
        search = catalogo.search(
            collections=[COLLECTION],
            datetime=f"{start}/{end}",
            bbox=bbox,
        )
        time.sleep(1)

        for item in search.items():
            items_by_id[item.id] = item   # dedupe items compartidos por ventanas de meses contiguos

    cluster_matchups = compute_matchups(cluster_points, list(items_by_id.values()))
    all_matchups.extend(cluster_matchups)

Searching months of LDS: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 70/70 [02:31<00:00,  2.16s/it]


In [20]:
len(all_matchups) 


3038

#### VAMOOO, LOS PIBEEEE
### OLE OLE OLE, OLE OLE OLE OLA

In [23]:
#all_matchups[:20]

In [26]:
all_matchups[:5]

[(<Item id=S2B_MSIL1C_20171105T135059_N0500_R024_T21HUD_20231013T190243>,
  297,
  Timedelta('1 days 20:19:00.973000')),
 (<Item id=S2A_MSIL1C_20171110T135111_N0500_R024_T21HUD_20230809T081726>,
  298,
  Timedelta('2 days 05:11:11.026000')),
 (<Item id=S2A_MSIL1C_20180109T135101_N0500_R024_T21HUD_20230917T191950>,
  356,
  Timedelta('0 days 05:51:01.026000')),
 (<Item id=S2A_MSIL1C_20180109T135101_N0500_R024_T21HUD_20230917T191950>,
  357,
  Timedelta('0 days 04:51:01.026000')),
 (<Item id=S2A_MSIL1C_20180109T135101_N0500_R024_T21HUD_20230917T191950>,
  358,
  Timedelta('0 days 04:06:01.026000'))]

In [49]:
gdf_copy = gdf.copy()

In [50]:
#gdf_copy["ventana"] = None
gdf_copy["pass_id"] = None
gdf_copy["delta"] = None

In [51]:
for matchup in all_matchups:
    item, index, delta = matchup
    gdf_copy.loc[index, "delta"] = str(delta)
    gdf_copy.loc[index, "pass_id"] = item.id

In [52]:
gdf_copy

,fecha,fuente,chla,grupo_nombre,estado_trofico,geometry,delta,pass_id
0,2017-01-02 00:00:00,OAN,8.9,RDP-MONTES,M,POINT (401164.986 6214585.025),2 days 13:50:52.026000,S2A_MSIL1C_20170104T135052_N0500_R024_T21HUC_2...
1,2017-01-02 00:00:00,GEMS,6.8,LDS,O,POINT (678380.002 6147956.963),3 days 10:27:37.974000,S2A_MSIL1C_20161229T133222_N0500_R081_T21HXB_2...
2,2017-01-02 00:00:00,GEMS,3.2,LDS,O,POINT (679045.962 6144090.994),3 days 10:27:37.974000,S2A_MSIL1C_20161229T133222_N0500_R081_T21HXB_2...
3,2017-01-02 00:00:00,OAN,4.4,RDP-MONTES,O,POINT (402233.023 6210401.038),2 days 13:50:52.026000,S2A_MSIL1C_20170104T135052_N0500_R024_T21HUC_2...
4,2017-01-02 00:00:00,OAN,5.9,RDP-MONTES,O,POINT (401555.004 6213022.975),2 days 13:50:52.026000,S2A_MSIL1C_20170104T135052_N0500_R024_T21HUC_2...
...,...,...,...,...,...,...,...,...
3084,2026-05-06 15:14:00,OAN,1.5,RN-UPM2,U,POINT (540411.19 6365961.336),0 days 01:31:38.976000,S2A_MSIL1C_20260506T134221_N0512_R124_T21HWD_2...
3085,2026-05-06 15:28:00,OAN,3.0,RN-UPM2,O,POINT (540435.203 6366112.119),0 days 01:45:38.976000,S2A_MSIL1C_20260506T134221_N0512_R124_T21HWD_2...
3086,2026-05-06 15:50:00,OAN,1.5,RN-UPM2,U,POINT (540534.657 6366265.697),0 days 02:07:38.976000,S2A_MSIL1C_20260506T134221_N0512_R124_T21HWD_2...
3087,2026-05-06 16:05:00,OAN,1.5,RN-UPM2,U,POINT (541076.43 6365906.268),0 days 02:22:38.976000,S2A_MSIL1C_20260506T134221_N0512_R124_T21HWD_2...


In [ ]:
gdf

In [41]:
all_matchups[0][0]

<Item id=S2B_MSIL1C_20171105T135059_N0500_R024_T21HUD_20231013T190243>

#### Guardamos los items de cada matchup (Unicos)

In [72]:
def matchup_items(matchups):
    items_by_ids = {}
    for m in matchups:
        item = m[0]
        items_by_ids[item.id] = item
    return items_by_ids     

In [75]:
items_by_ids = matchup_items(all_matchups)
all_items = [item for item in items_by_ids.values()]
ic = ItemCollection(all_items)
ic.save_object("../data/matchup_items.json")

In [76]:
gdf_copy.to_file("../data/registros_limpios_matchups.json", driver="GeoJSON")